# Crystallizer: symmetry rankings for argyrodite

1. crystallize the argyrodite MD run.
2. find out which part of the structure breaks the *F*-43*m* symmetry.


In [1]:
import plotly.io as pio

pio.renderers.default = 'plotly_mimetype+notebook_connected'

from gemdat import Trajectory
from gemdat.utils import VASPRUN

trajectory = Trajectory.from_vasprun(VASPRUN)
trajectory

Full Formula (Li48 P8 S40 Br8)
Reduced Formula: Li6PS5Br
abc   :  19.873726   9.919447   9.916454
angles:  90.214114  90.859135  89.950474
pbc   :       True       True       True
Constant lattice (True)
Sites (104)
Time steps (5000)

This is a short (5000 frame) Li&#8326;PS&#8325;Br run in a 2&times;1&times;1
supercell of the &#8776;9.9&nbsp;&#197; cubic conventional cell: 48 lithium ions diffusing
through a framework of PS&#8324; tetrahedra and S&sup2;&#8315;/Br&#8315; anions.

## Crystallize

[`Crystallizer.scan`][gemdat.crystallizer.Crystallizer.scan] reconstructs the structure
and ranks the possible space groups.
[`format`][gemdat.crystallizer.CrystallizerScan.format] prints that ranking, one row per
space group:

- **`symprec`** &mdash; the tightest tolerance (&#197;) in the scan that found the group;
- **`deviation`** &mdash; how far the structure really is from the group (&#197;): the
  largest distance any atom has to move to reach the closest structure with exactly that
  symmetry, a property of the structure rather than of the search;
- **`# orbits`** &mdash; the number of symmetry-distinct sites that are left;
- **`# hits`** &mdash; how many tolerances in the scan landed on this group.

### What `symprec` and `deviation` measure

Both are distances in &#197;, and both are a **maximum over atoms**, not an average or an RMS.

**`symprec`** is the tolerance handed to spglib (through pymatgen's `SpacegroupAnalyzer`).
spglib accepts a candidate symmetry operation only if the image of *every* atom lands within
`symprec` of its own atom of the same species (Cartesian distance, periodic boundaries, matched
one-to-one). A single atom further off rejects the operation, however well the rest fit.

In the ranking, `symprec` is the tightest scan that produced the group (by
default 40 log-spaced values from 0.01 to 0.5&nbsp;&#197;).

**`deviation`** is measured afterwards, once per space group, and involves no tolerance. It calculates the mean of positions (least-squares), and reports the atom with largest distance from this.
[`SymmetryAnalyzer.idealise`][gemdat.symmetry.SymmetryAnalyzer.idealise] returns that ideal
structure, and `to_cif(..., idealised=True)` writes it.

In [2]:
from gemdat import Crystallizer

resolution = 0.3  # Å
background_level = 0.2

cr = Crystallizer.from_trajectory(trajectory, floating_specie='Li', resolution=resolution)

scan = cr.scan(background_level=background_level)
result = scan.best()

print(scan.format())

symprec (Å)  deviation (Å)  angle dev (°)  space group  #  crystal system  # ops  # orbits  # hits
-----------  -------------  -------------  -----------  -  --------------  -----  --------  ------
0.01         0              1.42e-14       P1           1  triclinic       1      137       40    


P1, at every tolerance the scan tried (up to 0.5&nbsp;&#197;). Asking the scan for the
textbook space group explicitly with
[`at_spacegroup`][gemdat.crystallizer.CrystallizerScan.at_spacegroup] fails, and the error
lists what was found instead:

In [3]:
try:
    scan.at_spacegroup('F-43m')
except ValueError as error:
    print(error)

No symprec in this scan produced space group 'F-43m'. Found:
symprec (Å)  deviation (Å)  angle dev (°)  space group  #  crystal system  # ops  # orbits  # hits
-----------  -------------  -------------  -----------  -  --------------  -----  --------  ------
0.01         0              1.42e-14       P1           1  triclinic       1      137       40    


## Fixing the lattice to recover symmetry

The ranking is not limited to the crystallizer's output: a
[`SymmetryAnalyzer`][gemdat.symmetry.SymmetryAnalyzer] ranks any structure. So we can take
the structure apart and rank that, starting with the time-averaged host
[`framework`][gemdat.crystallizer.Crystallizer.framework].

Argyrodite's framework has three kinds of site: the phosphorus atoms, the four sulfur atoms
of each PS&#8324; tetrahedron, and "free" anion sites, which the S&sup2;&#8315; and
Br&#8315; ions share. We split the sulfur by its distance to the nearest phosphorus and add
the pieces one at a time.

In [4]:
from pymatgen.core import Structure

from gemdat import SymmetryAnalyzer

framework = cr.framework()

p_coords = framework.frac_coords[list(framework.indices_from_symbol('P'))]


def is_tetrahedral(site) -> bool:
    """Sulfur bonded to phosphorus (P-S is ~2.05 Å) belongs to a PS4 tetrahedron."""
    return site.specie.symbol == 'S' and (
        framework.lattice.get_all_distances(site.frac_coords, p_coords).min() < 2.5
    )


phosphorus = [site for site in framework if site.specie.symbol == 'P']
tetrahedra = [site for site in framework if is_tetrahedral(site)]
anions = [
    site for site in framework if site.specie.symbol in ('S', 'Br') and not is_tetrahedral(site)
]


def build(sites, species=None) -> Structure:
    return Structure(
        lattice=framework.lattice,
        species=species or [site.specie for site in sites],
        coords=[site.frac_coords for site in sites],
    )


pieces = {
    'P': build(phosphorus),
    'P + S (PS4)': build(phosphorus + tetrahedra),
    # Give every free anion site the same species: only *where* the anions are
    # matters to the symmetry, not whether a given site holds S or Br.
    'P + S (PS4) + anion sites': build(
        phosphorus + tetrahedra + anions,
        species=['P'] * len(phosphorus) + ['S'] * len(tetrahedra) + ['Br'] * len(anions),
    ),
    'full framework (S and Br distinct)': build(phosphorus + tetrahedra + anions),
}

rankings = {name: SymmetryAnalyzer(piece).rank() for name, piece in pieces.items()}

for name, ranking in rankings.items():
    found = ', '.join(
        f'{level.spacegroup_symbol} ({level.deviation:.2f} Å)' for level in ranking.candidates
    )
    print(f'{name:<36} {len(pieces[name]):>3} sites:  {found}')

P                                      8 sites:  Fm-3m (0.07 Å), P-1 (0.04 Å), P1 (0.00 Å)
P + S (PS4)                           40 sites:  F-43m (0.21 Å), P2 (0.09 Å), P1 (0.00 Å)
P + S (PS4) + anion sites             56 sites:  F-43m (0.20 Å), P2_1 (0.10 Å), P1 (0.00 Å)
full framework (S and Br distinct)    56 sites:  P1 (0.00 Å)


The candidates are listed highest symmetry first, each with the deviation it requires.
Phosphorus alone sits on a face-centred cubic lattice (*Fm*-3*m*), and adding the PS&#8324;
sulfur and the free anion sites brings it down to exactly argyrodite's *F*-43*m* (#216), at a
deviation of about 0.2&nbsp;&#197;. The full ranking of that framework shows the route up
from P1:

In [5]:
print(rankings['P + S (PS4) + anion sites'].format())

symprec (Å)  deviation (Å)  angle dev (°)  space group  #    crystal system  # ops  # orbits  # hits
-----------  -------------  -------------  -----------  ---  --------------  -----  --------  ------
0.01         0              1.42e-14       P1           1    triclinic       1      56        32    
0.247757     0.1018         0.214          P2_1         4    monoclinic      2      28        2     
0.302798     0.202          0.859          F-43m        216  cubic           64     4         6     


The *geometry* of the host is therefore cubic to within about 0.2&nbsp;&#197;.
What breaks it is the **chemistry** of the anion sites: once S&sup2;&#8315; and Br&#8315; are
told apart, no tolerance recovers any symmetry, because the two ions are distributed over the
anion sites of this cell without the order a space group would need. That is site disorder,
and no amount of MD sampling will average it away &mdash; a crystallographic model would
describe these sites with mixed S/Br occupancy instead.